# Filter sample points by MOLCA 2024 coverage and re-extract class values

Reads the 2019 photo-interpreted reference points GeoJSON, keeps only those that fall inside a valid MOLCA 2024 tile and on a non-nodata pixel, and re-samples the 2024 class value at each kept point for comparison against the original `molca_class`.

**To re-run for another region**: change `REGION` and the input/output paths in Cell 1. The rest of the notebook is region-agnostic.

## Cell 1 — Region paths (only thing to edit between regions)

In [12]:
import os
import platform
from pyproj import datadir
os.environ["PROJ_LIB"] = datadir.get_data_dir()
from rasterio.crs import CRS
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform as warp_transform

REGION = "Africa"  # Options: "Africa", "Amazon_Extension", "Siberia"

if platform.system() == "Windows":
    # Local Windows Environment
    PROJECT_DIR = r"d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design"
else:
    # Leonardo Supercomputer (Linux Environment)
    PROJECT_DIR = "." 

MOLCA_GPKG = os.path.join(PROJECT_DIR, "MOLCA", f"{REGION}_2024_UTM", f"molcx_{REGION.lower()}_2024_s2_exist.gpkg")


# Create output directory dynamically
os.makedirs(os.path.join(PROJECT_DIR, "Samples_2024"), exist_ok=True)

INPUT_BY_REGION = {
    "Africa": "africa_static_area_2019_photointerpreted_UniGE_merged.geojson",
    "Amazon_Extension": "amazonia_static_area_2019_photointerpreted_UniTN_merged.geojson",
    "Siberia": "siberia_static_area_2019_photointerpreted_UniPV_merged.geojson",
}

if REGION not in INPUT_BY_REGION:
    raise ValueError(f"Unsupported REGION '{REGION}'. Choose one of: {list(INPUT_BY_REGION)}")

input_name = INPUT_BY_REGION[REGION]

# Combine remaining paths using os.path.join to handle \ and / automatically
INPUT_GEOJSON = os.path.join(PROJECT_DIR, "Samples_2019", input_name)
MOLCA_TIFFS = os.path.join(PROJECT_DIR, "MOLCA", f"{REGION}_2024_UTM", "Tiffs")
LEGEND_CSV = os.path.join(PROJECT_DIR, "config", "MOLCA_legend.csv")

OUTPUT_PREFIX_BY_REGION = {
    "Africa": "africa",
    "Amazon_Extension": "amazonia",
    "Siberia": "siberia",
}

output_prefix = OUTPUT_PREFIX_BY_REGION[REGION]
OUTPUT_GEOJSON = os.path.join(PROJECT_DIR, "Samples_2024", f"{output_prefix}_static_area_2024_molca_filtered.geojson")

NODATA_CLASS = 255

print("Current OS:   ", platform.system())
print("Region:       ", REGION)
print("Input:        ", INPUT_GEOJSON)
print("MOLCA gpkg:   ", MOLCA_GPKG)
print("MOLCA tiffs:  ", MOLCA_TIFFS)
print("Legend:       ", LEGEND_CSV)
print("Output:       ", OUTPUT_GEOJSON)

Current OS:    Linux
Region:        Africa
Input:         ./Samples_2019/africa_static_area_2019_photointerpreted_UniGE_merged.geojson
MOLCA gpkg:    ./MOLCA/Africa_2024_UTM/molcx_africa_2024_s2_exist.gpkg
MOLCA tiffs:   ./MOLCA/Africa_2024_UTM/Tiffs
Legend:        ./config/MOLCA_legend.csv
Output:        ./Samples_2024/africa_static_area_2024_molca_filtered.geojson


## Cell 2 — Load and preview inputs

In [13]:
points = gpd.read_file(INPUT_GEOJSON)
print("Points shape:", points.shape, " CRS:", points.crs)
points.head()

Points shape: (8502, 3)  CRS: EPSG:4326


,molca_class,molca_label,geometry
0,20,Forest,POINT (27.74004 0.58563)
1,20,Forest,POINT (28.92861 1.19097)
2,20,Forest,POINT (28.25383 2.37497)
3,5,Shrubland,POINT (13.77574 7.56173)
4,5,Shrubland,POINT (17.57939 7.2322)


In [14]:
gdf = gpd.read_file(MOLCA_GPKG)
print("gpkg rows:", len(gdf), "  unique Tile_ID:", gdf['Tile_ID'].nunique(), "  CRS:", gdf.crs)
gdf.head()

gpkg rows: 2174   unique Tile_ID: 252   CRS: EPSG:4326


,Tile_ID,index_right,Existing_data_location,layer,tile_crs,tile_resolution,geometry
0,36NZN,9,/leonardo_work/IscrC_HSIGFM/qxu00001/ESA_CCI_H...,ESRI_Annual_LULC_2024,EPSG:32636,30,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ..."
1,36NZN,5,/leonardo_work/IscrC_HSIGFM/qxu00001/ESA_CCI_H...,ESRI_Annual_LULC_2024,EPSG:32636,30,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ..."
2,36NZN,42,/leonardo_work/IscrC_HSIGFM/qxu00001/ESA_CCI_H...,GSW_Yearly_2021,EPSG:32636,30,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ..."
3,36NZN,48,/leonardo_work/IscrC_HSIGFM/qxu00001/ESA_CCI_H...,GWL_FCS30D_2022,EPSG:32636,30,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ..."
4,36NZN,35,/leonardo_work/IscrC_HSIGFM/qxu00001/ESA_CCI_H...,GLC_FCS30D_2022,EPSG:32636,30,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ..."


In [15]:
legend_df = pd.read_csv(LEGEND_CSV)
legend_df

,Class code,Label
0,5,Shrubland
1,7,Grassland
2,8,Cropland
3,13,Built-up
4,12,Bareland
5,16,Permanent ice and snow
6,15,Water
7,9,Wetland
8,11,Lichens and mosses
9,20,Forest


## Cell 3 — Peek at one tile to verify the legend

Confirm the unique class values found in the raster are a subset of the legend's `Class code` column. If unfamiliar codes appear, stop and investigate.

In [17]:
sample_tile = next((tile_id for tile_id in gdf['Tile_ID'].dropna().astype(str).unique() if os.path.exists(os.path.join(MOLCA_TIFFS, f"MOLCA_{tile_id}_F.tif"))), None)
if sample_tile is None:
    raise FileNotFoundError("No matching MOLCA GeoTIFF was found for the Tile_ID values loaded in gdf.")
print("Sample tile:    ", sample_tile)
with rasterio.open(os.path.join(MOLCA_TIFFS, f"MOLCA_{sample_tile}_F.tif")) as src:
    print("CRS:            ", src.crs)
    print("Resolution:     ", src.res)
    print("Nodata declared:", src.nodata)
    print("Bounds:         ", src.bounds)

print("\nScanning all tiles to find all unique classes in 2024 rasters...")
tile_classes = {}
for tile_id in gdf['Tile_ID'].dropna().unique():
    tif_path = os.path.join(MOLCA_TIFFS, f"MOLCA_{tile_id}_F.tif")
    if os.path.exists(tif_path):
        with rasterio.open(tif_path) as src:
            u = np.unique(src.read(1))
        tile_classes[tile_id] = set(int(v) for v in u if v != NODATA_CLASS)

all_2024_classes = sorted(list(set().union(*tile_classes.values())))
print("Unique class values across ALL tiles in this region:")
label_map = dict(zip(legend_df['Class code'], legend_df['Label']))
for c in all_2024_classes:
    print(f" {c}: {label_map.get(c, 'Unknown')}")

Sample tile:     36NZN
CRS:             EPSG:32636
Resolution:      (30.0, 30.0)
Nodata declared: 255.0
Bounds:          BoundingBox(left=799980.0, bottom=690240.0, right=909780.0, top=800040.0)

Scanning all tiles to find all unique classes in 2024 rasters...
Unique class values across ALL tiles in this region:
 7: Grassland
 8: Cropland
 9: Wetland
 12: Bareland
 13: Built-up
 15: Water
 20: Forest


## Cell 4 — Collapse gpkg to unique tiles + verify .tif exists

The gpkg lists each tile once per source dataset (≈ 2,174 rows for ~252 unique tiles). Reduce to one polygon per `Tile_ID`, then check that each tile's `.tif` actually exists on disk.

In [18]:
tiles = gdf[['Tile_ID', 'geometry']].drop_duplicates('Tile_ID').reset_index(drop=True)
tiles['tif_path'] = tiles['Tile_ID'].apply(lambda t: os.path.join(MOLCA_TIFFS, f"MOLCA_{t}_F.tif"))
tiles['tif_exists'] = tiles['tif_path'].apply(os.path.exists)
print("Tiles in gpkg:", len(tiles), "  with .tif on disk:", int(tiles['tif_exists'].sum()))
missing = tiles.loc[~tiles['tif_exists'], 'Tile_ID'].tolist()
if missing:
    print("Tiles listed in gpkg but missing on disk:", missing)
tiles = tiles[tiles['tif_exists']].drop(columns='tif_exists').reset_index(drop=True)
tiles.head()

Tiles in gpkg: 252   with .tif on disk: 252


,Tile_ID,geometry,tif_path
0,36NZN,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36NZN_F.tif
1,36NZP,"POLYGON ((35.72206 8.13291, 36.71716 8.12502, ...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36NZP_F.tif
2,36PZA,"POLYGON ((35.78245 14.45638, 36.79951 14.44216...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36PZA_F.tif
3,36PZB,"POLYGON ((35.79408 15.35961, 36.81538 15.34446...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36PZB_F.tif
4,36PZQ,"POLYGON ((35.72851 9.03659, 36.72596 9.02781, ...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36PZQ_F.tif


## Cell 5 — Spatial join: point → tile

Each surviving point gets the `Tile_ID` of the tile that contains it. Rows where `Tile_ID` is NaN are outside MOLCA 2024 coverage and will be dropped.

In [19]:
tiles = tiles.to_crs(points.crs)
joined = gpd.sjoin(points, tiles[['Tile_ID', 'geometry']], predicate="within", how="left")
# Drop duplicate matches from tile-edge overlaps (rare); keep first.
joined = joined[~joined.index.duplicated(keep='first')]
n_outside = int(joined['Tile_ID'].isna().sum())
print(f"Points outside MOLCA 2024 coverage: {n_outside} / {len(joined)}")
joined.head()

Points outside MOLCA 2024 coverage: 5061 / 8502


,molca_class,molca_label,geometry,index_right,Tile_ID
0,20,Forest,POINT (27.74004 0.58563),NaN,NaN
1,20,Forest,POINT (28.92861 1.19097),NaN,NaN
2,20,Forest,POINT (28.25383 2.37497),NaN,NaN
3,5,Shrubland,POINT (13.77574 7.56173),NaN,NaN
4,5,Shrubland,POINT (17.57939 7.2322),NaN,NaN


In [20]:
# Show a few of the points that fell outside coverage, for sanity
joined[joined['Tile_ID'].isna()].head()

,molca_class,molca_label,geometry,index_right,Tile_ID
0,20,Forest,POINT (27.74004 0.58563),NaN,NaN
1,20,Forest,POINT (28.92861 1.19097),NaN,NaN
2,20,Forest,POINT (28.25383 2.37497),NaN,NaN
3,5,Shrubland,POINT (13.77574 7.56173),NaN,NaN
4,5,Shrubland,POINT (17.57939 7.2322),NaN,NaN


## Cell 6 — Raster sampling per tile

For each tile group: open the GeoTIFF once, reproject the group's lon/lat into the tile's UTM CRS, batch-sample pixel values.

In [21]:
joined['molca_class_2024'] = np.nan
inside = joined.dropna(subset=['Tile_ID'])
n_tiles = inside['Tile_ID'].nunique()
print(f"Sampling {len(inside)} points across {n_tiles} tiles...")

for tile_id, grp in inside.groupby('Tile_ID'):
    lons = grp.geometry.x.tolist()
    lats = grp.geometry.y.tolist()
    with rasterio.open(os.path.join(MOLCA_TIFFS, f"MOLCA_{tile_id}_F.tif")) as src:
        xs, ys = warp_transform("EPSG:4326", src.crs, lons, lats)
        vals = [v[0] for v in src.sample(list(zip(xs, ys)))]
    joined.loc[grp.index, 'molca_class_2024'] = vals

print("Done.  Unique sampled values:", sorted(joined['molca_class_2024'].dropna().unique().tolist()))

Sampling 3441 points across 229 tiles...
Done.  Unique sampled values: [7.0, 8.0, 9.0, 12.0, 13.0, 15.0, 20.0, 255.0]


## Cell 7 — Filter nodata, attach label, compare, write GeoJSON

In [22]:
n_nodata = int(((joined['molca_class_2024'] == NODATA_CLASS)).sum())

kept = joined[joined['molca_class_2024'].notna() & (joined['molca_class_2024'] != NODATA_CLASS)].copy()
kept['molca_class_2024'] = kept['molca_class_2024'].astype(int)

label_map = dict(zip(legend_df['Class code'], legend_df['Label']))
kept['molca_label_2024'] = kept['molca_class_2024'].map(label_map)
kept['class_agrees'] = kept['molca_class'] == kept['molca_class_2024']

out_cols = ['molca_class', 'molca_label', 'molca_class_2024', 'molca_label_2024', 'class_agrees', 'Tile_ID', 'geometry']
out = kept[out_cols].copy()
out = gpd.GeoDataFrame(out, geometry='geometry', crs=points.crs)

os.makedirs(os.path.dirname(OUTPUT_GEOJSON), exist_ok=True)
if os.path.exists(OUTPUT_GEOJSON):
    os.remove(OUTPUT_GEOJSON)
out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
print("Wrote:", OUTPUT_GEOJSON)
out.head()

Wrote: ./Samples_2024/africa_static_area_2024_molca_filtered.geojson


,molca_class,molca_label,molca_class_2024,molca_label_2024,class_agrees,Tile_ID,geometry
141,5,Shrubland,7,Grassland,False,35PPS,POINT (28.68598 15.29629)
150,5,Shrubland,20,Forest,False,35PRR,POINT (29.85248 14.30307)
152,5,Shrubland,20,Forest,False,35PQS,POINT (29.15286 14.58446)
185,5,Shrubland,12,Bareland,False,37PES,POINT (39.73347 15.01841)
188,5,Shrubland,12,Bareland,False,37PHP,POINT (42.4963 12.39016)


## Cell 8 — Summary

In [23]:
total = len(points)
kept_n = len(kept)
agree_n = int(kept['class_agrees'].sum())
agree_pct = 100.0 * agree_n / kept_n if kept_n else 0.0
assert n_outside + n_nodata + kept_n == total, "Counts do not add up"

print(f"Total input:                {total}")
print(f"Dropped (outside MOLCA):    {n_outside}")
print(f"Dropped (nodata, class 255):{n_nodata}")
print(f"Kept:                       {kept_n}")
print(f"Agreement with 2019 label:  {agree_n} / {kept_n}  ({agree_pct:.1f}%)")

Total input:                8502
Dropped (outside MOLCA):    5061
Dropped (nodata, class 255):2249
Kept:                       1192
Agreement with 2019 label:  729 / 1192  (61.2%)


In [24]:
# Count points in each class (2024 classification)
kept_by_class = kept['molca_class_2024'].value_counts().sort_index()
print("Points in each class (2024):")
print(kept_by_class)
print(f"\nTotal: {kept_by_class.sum()}")

Points in each class (2024):
molca_class_2024
7      55
8     194
9      34
12    192
13     26
15    178
20    513
Name: count, dtype: int64

Total: 1192


## Cell 9 — Per-class tile coverage vs. tiles hosting the 1,192 filtered points

Restricted to the 7 classes actually present in MOLCA 2024 (7, 8, 9, 12, 13, 15, 20). For each class:
- **tiles_with_class**: tiles (out of 252) whose MOLCA 2024 raster contains at least one valid pixel of that class
- **tiles_with_kept_pts**: of those tiles, how many host at least one of the 1,192 kept points whose 2024-resampled class equals this class

In [ ]:
# tile_classes was already built in Cell 3 by scanning all tiles

# Tiles that host at least one kept point, grouped by the point's 2024-resampled class
tiles_with_kept_by_class = (
    kept.groupby('molca_class_2024')['Tile_ID']
        .apply(lambda s: set(s.unique()))
        .to_dict()
)

# Only the classes actually present in the kept points
classes_present_2024 = sorted(kept['molca_class_2024'].unique().tolist())
label_map = dict(zip(legend_df['Class code'], legend_df['Label']))

rows = []
for code in classes_present_2024:
    tiles_with_class = {t for t, classes in tile_classes.items() if code in classes}
    tiles_with_kept  = tiles_with_kept_by_class.get(code, set())
    rows.append({
        'class_code': code,
        'label': label_map.get(code, '?'),
        'tiles_with_class': len(tiles_with_class),
        'tiles_with_kept_pts': len(tiles_with_kept),
    })

summary = pd.DataFrame(rows)
print(f"Total tiles considered: {len(tiles)}")
print(f"Kept points:            {len(kept)}  across {kept['Tile_ID'].nunique()} tiles")
print(summary.to_string(index=False))

## Cell 10 — Distribution of kept points per tile, by class

For each 2024 class, bucket tiles by how many kept points they host: 1–2, 3–5, 6–9, 10+. Also report the max points in a single tile.

In [ ]:
# Points-per-tile counts for each 2024 class, then bucket
bins   = [0, 2, 5, 9, np.inf]
labels = ['1-2', '3-5', '6-9', '10+']

rows = []
for code in sorted(kept['molca_class_2024'].unique().tolist()):
    sub = kept[kept['molca_class_2024'] == code]
    per_tile = sub.groupby('Tile_ID').size()  # points per tile (only tiles with >=1 point)
    buckets  = pd.cut(per_tile, bins=bins, labels=labels).value_counts().reindex(labels, fill_value=0)
    rows.append({
        'class_code': code,
        'label':      label_map.get(code, '?'),
        'points':     int(per_tile.sum()),
        'tiles':      int(per_tile.size),
        '1-2':        int(buckets['1-2']),
        '3-5':        int(buckets['3-5']),
        '6-9':        int(buckets['6-9']),
        '10+':        int(buckets['10+']),
        'max_per_tile': int(per_tile.max()),
        'median_per_tile': float(per_tile.median()),
    })

dist = pd.DataFrame(rows)
print(dist.to_string(index=False))

 class_code     label  points  tiles  1-2  3-5  6-9  10+  max_per_tile  median_per_tile
          7 Grassland      55     27   23    2    1    1            11              1.0
          8  Cropland     194     36   23    6    3    4            48              2.0
          9   Wetland      34     12    8    2    1    1            11              1.0
         12  Bareland     192     28   13    9    2    4            54              3.0
         13  Built-up      26      7    5    0    1    1            15              1.0
         15     Water     178     77   59   11    5    2            12              1.0
         20    Forest     513     81   43   20    7   11           138              2.0
